# Atlas fix-plan example

This notebook uses a tiny synthetic Atlas dataset and a JSON fix plan to show the public Woodpecker plan API with bundled plugin fixes.

In [ ]:
import json

import woodpecker_atlas_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import integration_plan_path, make_atlas

Create a deterministic Atlas-like dataset with two issues: missing `project_id` metadata and overly strong compression settings.

In [ ]:
dataset = make_atlas(missing=["project_id"], seed=7)
dataset["pr"].encoding["complevel"] = 5

dataset

Load the Atlas fix-plan document and inspect its content.

The plan matches `atlas*.nc`-style inputs and selects the bundled Atlas plugin fixes.

In [ ]:
plan_path = integration_plan_path("atlas_basic_plan.json")

plan_path

In [ ]:
plan_json = json.loads(plan_path.read_text(encoding="utf-8"))
plan_json

In [ ]:
findings = woodpecker.plan.check(dataset, plan_path)
findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
dry_run = woodpecker.plan.fix(dataset, plan_path, dry_run=True)

dry_run.stats, dataset.attrs.get("project_id"), dataset["pr"].encoding["complevel"]

Apply the same plan in memory with `dry_run=False`.

In [ ]:
write = woodpecker.plan.fix(dataset, plan_path, dry_run=False)

(
    write.stats,
    dataset.attrs["project_id"],
    dataset["pr"].encoding["complevel"],
    dataset["pr"].encoding["zlib"],
    dataset["pr"].encoding["shuffle"],
)

In [ ]:
recheck = woodpecker.plan.check(dataset, plan_path)
bool(recheck)